In [12]:
#!/usr/bin/env python
import numpy as np 
import pandas as pd 
import mediapipe as mp 
import csv
import tensorflow as tf 
import cv2 as cv 
import copy


ImportError: attempted relative import with no known parent package

In [16]:
import sys
import os

# Specify the root directory manually
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Add the parent directory to sys.path
sys.path.append(project_root)

# Now you can import from the 'src' directory
from src import main

NameError: name 'null' is not defined

In [2]:
model_path = "../Saved Models/model.hdf5"
model = tf.keras.models.load_model(model_path)

In [3]:
def predict(landmarks):
    input_data = np.expand_dims(landmarks, axis=0)
    mode = model.predict(input_data)
    result_index = np.argmax(np.squeeze(mode))
    return (result_index,np.squeeze(mode)[result_index])

In [4]:
def get_landmarks(image,hand_landmarks):
    landmark_list = []
    img_height,img_width = image.shape[0],image.shape[1]
    for landmarks in hand_landmarks.landmark:
        x = min(int(landmarks.x*img_width),img_width-1)
        y = min(int(landmarks.y*img_height),img_height-1)
        landmark_list.append([x,y])
    return landmark_list

def process_landmarks(landmarks):
    relative_x,relative_y = 0,0
    temp_landmarks = copy.deepcopy(landmarks)
    final_landmarks = []
    for index, point in enumerate(temp_landmarks):
        if index == 0:
            relative_x,relative_y = point[0],point[1]
        temp_landmarks[index][0] = point[0] - relative_x
        temp_landmarks[index][1] = point[1] - relative_y
        final_landmarks.append(temp_landmarks[index][0])
        final_landmarks.append(temp_landmarks[index][1])
    max_value = max(list(map(abs,final_landmarks)))

    def normalize(n):
        return n/max_value

    final_landmarks = list(map(normalize,final_landmarks))

    return final_landmarks

In [5]:
def main():
    cap = cv.VideoCapture(0)
    mp_hands = mp.solutions.hands
    mp_draw = mp.solutions.drawing_utils
    mp_draw_styles = mp.solutions.drawing_styles

    if not cap.isOpened():
        print("Error opening camera")
        exit()

    with mp_hands.Hands(
    model_complexity=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
    ) as hands:
        while True:
            ret,frame = cap.read()
            if not ret :
                print("error capturing")
                break
            else:
                image = cv.flip(frame,1)
                debug_image = cv.cvtColor(image,cv.COLOR_BGR2RGB)
                result = hands.process(debug_image)
                if result.multi_hand_landmarks:
                    for hand_landmarks in result.multi_hand_landmarks:
                        mp_draw.draw_landmarks(image,hand_landmarks,
                        mp_hands.HAND_CONNECTIONS,
                        mp_draw_styles.get_default_hand_landmarks_style(),
                        mp_draw_styles.get_default_hand_connections_style()
                        )
                        landmarks_list = get_landmarks(frame,hand_landmarks)
                        processed_lanamrks = process_landmarks(landmarks_list)
                        mode = predict(processed_lanamrks)
                        print("mode: ",mode[0])
                        print("prob: ",mode[1])
                cv.imshow("Camera Frame",image)
                key  = cv.waitKey(1) & 0xFF
                # Break the loop if 'q' is pressed
                if key == ord('q'):
                    print("Exiting...")
                    break

    cap.release()
    cv.destroyAllWindows()

In [6]:
if __name__ == "__main__":
    main()
    


I0000 00:00:1734029909.385037   76435 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1734029909.389240   76509 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 21.2.6), renderer: Mesa Intel(R) UHD Graphics (CML GT2)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


1/1 [==============================] - 0s 121ms/step
mode:  0
prob:  0.95699316
1/1 [==============================] - 0s 13ms/step
mode:  0
prob:  0.99494064
1/1 [==============================] - 0s 15ms/step
mode:  0
prob:  0.9968117
1/1 [==============================] - 0s 14ms/step
mode:  0
prob:  0.9969382
1/1 [==============================] - 0s 14ms/step
mode:  0
prob:  0.99744797
1/1 [==============================] - 0s 13ms/step
mode:  0
prob:  0.9966941
1/1 [==============================] - 0s 14ms/step
mode:  0
prob:  0.995269
1/1 [==============================] - 0s 13ms/step
mode:  0
prob:  0.98938024
1/1 [==============================] - 0s 13ms/step
mode:  0
prob:  0.99584335
1/1 [==============================] - 0s 14ms/step
mode:  0
prob:  0.9959125
1/1 [==============================] - 0s 14ms/step
mode:  0
prob:  0.9960867
1/1 [==============================] - 0s 13ms/step
mode:  0
prob:  0.99693847
1/1 [==============================] - 0s 14ms/step
mode: 